In [ ]:
""" Read in coparative datasets and sanity check their labels """
# import pandas as pd
# comparative_datasets = ['original', 'cleaned', 'augmented']
# for dataset in comparative_datasets:
#     data = pd.read_csv(f'comparative_datasets/{dataset}/combined_dataset.csv')
#     print(f"{dataset} dataset shape: {data.shape}")
#     print(data['label'].value_counts(normalize=True))

original dataset shape: (218367, 5)
label
1    0.669043
0    0.330957
Name: proportion, dtype: float64
cleaned dataset shape: (146097, 5)
label
1    1.0
Name: proportion, dtype: float64
cleaned_doubled dataset shape: (292194, 5)
label
1    1.0
Name: proportion, dtype: float64


In [9]:
"""For building the pickle files for each grade from original CSV files"""
from sklearn.model_selection import train_test_split
from collections import defaultdict
from datasets import DatasetDict, Dataset
import pickle
import pandas as pd

def split_grade_pkl_from_full_csv(dataset_type: str):
    print(f"Processing dataset type: {dataset_type}")
    all_data = pd.read_csv(f'comparative_datasets/{dataset_type}/combined_dataset.csv')
    print(f"Loaded data shape: {all_data.shape}")

    grades = all_data.groupby('abs_tgt_FKGL_Grade')

    grade_dict = defaultdict()
    split_dict = defaultdict()
    for grade in grades:
        grade_dict[grade[0]] = grade[1].reset_index(drop=True)

    for grade in grade_dict.keys():
        if grade in {0, 1, 13}:
            print(f"Skipping grade {grade} due to insufficient data.")
            continue
        train, test_validate = train_test_split(grade_dict[grade], test_size=1024, random_state=42,)
        train_size = (train.shape[0] // 32 ) * 32
        assert train_size % 32 == 0, "Train size must be a multiple of 32"
        train = train.iloc[:train_size]  # ensure train size is a multiple of 32
        print(f"grade {grade} - Train size: {train.shape[0]}, Test/validate size: {test_validate.shape[0]}")
        validate, test = train_test_split(test_validate, test_size=0.5, random_state=42)
        split_dict[grade] = {'train': train.reset_index(drop=True), 'validate': validate.reset_index(drop=True), 'test': test.reset_index(drop=True)}

    # convert dataframes to datasets and save to pickle format
    for grade in split_dict.keys():
        train_data = Dataset.from_pandas(split_dict[grade]['train'])
        validate_data = Dataset.from_pandas(split_dict[grade]['validate'])
        test_data = Dataset.from_pandas(split_dict[grade]['test'])
        dataset_dict = DatasetDict({'train': train_data, 'validate': validate_data, 'test': test_data})
        # print(dataset_dict)
        print(f"Dataset for grade {grade} created with lengths: train={len(train_data)}, validate={len(validate_data)}, test={len(test_data)}")
        with open(f'comparative_datasets/{dataset_type}/grade{grade:02}/{grade}.pkl', 'wb') as f:
            pickle.dump(dataset_dict, f)
    print(f"All datasets for {dataset_type} saved successfully.")

for dataset_type in ['original', 'cleaned', 'augmented']:
    split_grade_pkl_from_full_csv(dataset_type)

Processing dataset type: original
Loaded data shape: (218367, 5)
Skipping grade 0 due to insufficient data.
Skipping grade 1 due to insufficient data.
grade 2 - Train size: 7328, Test/validate size: 1024
grade 3 - Train size: 9632, Test/validate size: 1024
grade 4 - Train size: 19968, Test/validate size: 1024
grade 5 - Train size: 12832, Test/validate size: 1024
grade 6 - Train size: 24544, Test/validate size: 1024
grade 7 - Train size: 17568, Test/validate size: 1024
grade 8 - Train size: 22688, Test/validate size: 1024
grade 9 - Train size: 15040, Test/validate size: 1024
grade 10 - Train size: 17504, Test/validate size: 1024
grade 11 - Train size: 8128, Test/validate size: 1024
grade 12 - Train size: 9952, Test/validate size: 1024
Skipping grade 13 due to insufficient data.
Dataset for grade 2 created with lengths: train=7328, validate=512, test=512
Dataset for grade 3 created with lengths: train=9632, validate=512, test=512
Dataset for grade 4 created with lengths: train=19968, val

In [10]:
"""Import pickle, convert back to dataframe, export as json in alpaca format"""
import os
from datasets import DatasetDict
import pickle
import pandas as pd
from huggingface_hub import login, HfApi, upload_folder
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("HUGGINGFACE_HUB_TOKEN")
login(token=token)

def reimport_and_save_as_json(grade, dataset_type):
    # Load the pickled dataset

    #system_prompt = "Please rewrite the following sentence to make it easily understandable by students in Grade {tgt_ideal_Grade}. Ensure that the rewritten sentence is grammatically correct, fluent, and retains the core message of the original sentence without changing its meaning."
    instruction_prompt = "Rewrite this Input sentence to make it easily understandable by students in Grade {tgt_ideal_Grade}"# while preserving the meaning: Please note, if the initial rewrite does not meet the specified grade level, you are encouraged to modify and regenerate the output until the criteria are satisfactorily met. The final output should only include the last, correct version of the rewritten sentence(s)."
    
    with open(f'comparative_datasets/{dataset_type}/grade{grade:02}/{grade}.pkl', 'rb') as file:
        data = pickle.load(file)
    
    dataset = DatasetDict()
    
    for split_str in ["train", "validate", "test"]:
        split = pd.DataFrame(data[split_str])
        split['input'] = split['src'].apply(lambda x: x.strip())
        split['output'] = split['tgt'].apply(lambda x: x.strip())

        split['instruction'] = split.apply(lambda x: instruction_prompt.format(tgt_ideal_Grade=grade), axis=1)
  
        split['src_grade'] = split['abs_src_FKGL_Grade']
        split['tgt_grade'] = split['abs_tgt_FKGL_Grade']

        split = split[['input', 'output', 'instruction', 'src_grade', 'tgt_grade', 'label']]
        
        dataset[split_str] = Dataset.from_pandas(split)
    
    # Save as Alpaca-formatted JSON
    path = f'comparative_datasets/{dataset_type}/grade{grade:02}/'
    assert os.path.exists(path), f"Path {path} does not exist."

    for split_str in ["train", "validate", "test"]:
        pd.DataFrame(dataset[split_str]).to_json(f'{path}/{split_str}.json', orient='records', indent=2)
    
    # # Upload JSON files directly to hub to maintain JSON format
    # api = HfApi()
    # repo_id = f'williamplacroix/graded_wikilarge'
    
    # # # # Create repo if it doesn't exist
    # # try:
    # #     api.create_repo(repo_id=repo_id, private=True, exist_ok=True, repo_type="dataset")
    # # except Exception as e:
    # #     print(f"Repo creation note: {e}")
    
    # # Upload each JSON file
    # for split_str in ["train", "validate", "test"]:
    #     # try:
    #     #     api.delete_file(
    #     #         path_in_repo=f'{split_str}.json',
    #     #         repo_id=repo_id,
    #     #         repo_type="dataset",  # or "model"
    #     #     )
    #     # except Exception as e:
    #     #     print(f"nothing to delete: {e}")
    #     api.upload_file(
    #         path_or_fileobj=f'{path}/{split_str}.json',
    #         path_in_repo=f'{split_str}.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update {split_str} split",
    #         repo_type="dataset"
    #     )
    
    print(f"Grade {grade} processed and saved to {path}")

from tqdm import tqdm
# test with grade 8
for dataset_type in ['original', 'cleaned', 'augmented']:
    print(f"Processing dataset type: {dataset_type}")
    for grade in tqdm({2,3,4,5,6,7,8,9,10,11,12}):
        reimport_and_save_as_json(grade, dataset_type)
    print(f"All grades for {dataset_type} processed and saved successfully.")



100%|██████████| 11/11 [00:30<00:00,  2.77s/it]


In [ ]:
""" Sample from each grade for building baseline validation and test sets """

from datasets import load_dataset
def combine_and_split_baseline(dataset_type) -> None:
    print(f"Combining and splitting baseline for dataset type: {dataset_type}")
    val_sets = []
    test_sets = []
    train_sets = []
    for grade in tqdm(range(2, 13)):
        # with open (f'grade_{grade}/val.json') as f:
        #     grade_data = json.load(f)
        # dataset = load_dataset(f"williamplacroix/graded_wikilarge",
        #                         data_dir="cleaned/grade05",
        #                         split="train")

        # train_dataframe = pd.DataFrame(dataset['train'])
        # val_dataframe = pd.DataFrame(dataset['validate'])
        # test_dataframe = pd.DataFrame(dataset['test'])
        #grade_dataframe = pd.DataFrame(dataset)
        train_dataframe = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/train.json')
        val_dataframe = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/validate.json')
        test_dataframe = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/test.json')

        train_dataframe['label'] = grade
        val_dataframe['label'] = grade
        test_dataframe['label'] = grade
        
        train_sets.append(train_dataframe)
        val_sets.append(val_dataframe)
        test_sets.append(test_dataframe)
        # print(f"Grade {grade} - Train: {len(train_dataframe)}, Val: {len(val_dataframe)}, Test: {len(test_dataframe)}")

    combined_train = pd.concat(train_sets, ignore_index=True)
    combined_val = pd.concat(val_sets, ignore_index=True)
    combined_test = pd.concat(test_sets, ignore_index=True)

    # train_val_sample, _ = train_test_split(combined_train, train_size=len(combined_val), random_state=42, stratify=combined_train['label'])
    # train_test_sample, _ = train_test_split(combined_train, train_size=len(combined_test), random_state=42, stratify=combined_train['label'])
    
    val, _ = train_test_split(combined_val, train_size=512, random_state=42)#, stratify=train_val_sample['label'])
    test, _ = train_test_split(combined_test, train_size=512, random_state=42)#, stratify=train_test_sample['label'])
    # print(val['label'].value_counts())
    # print(test['label'].value_counts())
    print(val.shape)
    print(test.shape)
    # try:
    #     os.mkdir(f'./baseline')
    # except:
    #     pass 
    pd.DataFrame(val).to_json(f'comparative_datasets/{dataset_type}/baseline/validate.json', orient='records', indent=2)
    pd.DataFrame(test).to_json(f'comparative_datasets/{dataset_type}/baseline/test.json', orient='records', indent=2)
    
    print(f"Baseline validation and test sets for {dataset_type} saved successfully.")
    # datasets.Dataset.from_pandas(val).push_to_hub("williamplacroix/wikilarge_baseline_val")
    # datasets.Dataset.from_pandas(test).push_to_hub("williamplacroix/wikilarge_baseline_test")
    # api = HfApi()
    # repo_id = f'williamplacroix/wikilarge_baseline_alpaca'
    # api.upload_file(
    #         path_or_fileobj='./data/wikilarge/cleaned_graded_splits/wikilarge_baseline_alpaca/validate.json',
    #         path_in_repo='validate.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update validate split",
    #         repo_type="dataset"
    #     )
    # api.upload_file(
    #         path_or_fileobj='./data/wikilarge/cleaned_graded_splits/wikilarge_baseline_alpaca/test.json',
    #         path_in_repo='test.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update test split",
    #         repo_type="dataset"
    #     )
    
for dataset_type in ['cleaned', 'original', 'augmented']:
    combine_and_split_baseline(dataset_type)    

Combining and splitting baseline for dataset type: cleaned


 18%|█▊        | 2/11 [00:00<00:00, 15.30it/s]

Grade 2 - Train: 2944, Val: 512, Test: 512
Grade 3 - Train: 3744, Val: 512, Test: 512
Grade 4 - Train: 11264, Val: 512, Test: 512


 36%|███▋      | 4/11 [00:00<00:00, 11.42it/s]

Grade 5 - Train: 8416, Val: 512, Test: 512
Grade 6 - Train: 17632, Val: 512, Test: 512


...

Grade 8 - Train: 31456, Val: 512, Test: 512
Grade 9 - Train: 22016, Val: 512, Test: 512


 91%|█████████ | 10/11 [00:01<00:00,  5.35it/s]

Grade 10 - Train: 32576, Val: 512, Test: 512
Grade 11 - Train: 18016, Val: 512, Test: 512


100%|██████████| 11/11 [00:01<00:00,  5.89it/s]

Grade 12 - Train: 23584, Val: 512, Test: 512
label
8     58
12    58
3     54
7     54
5     47
11    45
9     43
6     40
4     39
10    38
2     36
Name: count, dtype: int64
label
8     58
12    58
3     54
7     54
5     47
11    45
9     43
6     40
4     39
10    38
2     36
Name: count, dtype: int64
(512, 6)
(512, 6)
Baseline validation and test sets for augmented saved successfully.


In [14]:
""" Upload all JSON files to Hugging Face Hub """

api = HfApi()
repo_id = f'williamplacroix/graded_wikilarge'

upload_folder(
        repo_id=repo_id,
        repo_type="dataset",
        folder_path="comparative_datasets",
        path_in_repo="",                 # keep same structure
        allow_patterns=["**/*.json"],
        commit_message="Initial upload of 3 variations × 11 grades (+baseline) × 3 splits",
    )

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]























































































































































































































































































































































































































































































































train.json: 100%|██████████| 10.9M/10.9M [00:09<00:00, 1.16MB/s]











train.json: 100%|██████████| 11.5M/11.5M [00:09<00:00, 1.19MB/s]








Upload 4 LFS files:  25%|██▌       | 1/4 [00:09<00:29,  9.88s/it]









train.json: 100%|██████████| 12.8M/12.8M [00:10<00:00, 1.26MB/s]



train.json: 100%|██████████| 14.0M/14.0M [00:10<00:00, 1.33MB/s]]
Upload 4 LFS files: 100%|██████████| 4/4 [00:10<00:00,  2.70s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/williamplacroix/graded_wikilarge/commit/c4bdddf2ddfbece58774782d743771415c13f82e', commit_message='Initial upload of 3 variations × 11 grades (+baseline) × 3 splits', commit_description='', oid='c4bdddf2ddfbece58774782d743771415c13f82e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/williamplacroix/graded_wikilarge', endpoint='https://huggingface.co', repo_type='dataset', repo_id='williamplacroix/graded_wikilarge'), pr_revision=None, pr_num=None)

In [ ]:
""" Update a single split later: """

from huggingface_hub import upload_file

group = "cleaned"
grade = "grade05"
split = "train"

upload_file(
    path_or_fileobj=f"comparative_datasets/{group}/{grade}/{split}.json",
    path_in_repo=f"{group}/{grade}/{split}.json",
    repo_id="williamplacroix/graded_wikilarge",
    repo_type="dataset",
    commit_message=f"Update {group} {grade} {split}"
)